# The three-number comparator -- appendix summary figure (frozen copy)

Pure plotting pass over cached results from `explore_multicompare.ipynb` (n3), `explore_multicompare2.ipynb` (n2), `explore_comparatorfig.ipynb`/`makefig_comparator.ipynb` (the two-number comparator neurons), and this session's L19/L20 read-out-token work. No model, no recomputation.

In [ ]:
import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D            # noqa: F401  (registers the 3d projection)

RESULTS_DIR, FIGS_DIR = "results", "figs"
os.makedirs(FIGS_DIR, exist_ok=True)
REGIME = "2digit"

## Load cached results

In [ ]:
def load(name):
    return dict(np.load(f'{RESULTS_DIR}/{name}_{REGIME}.npz', allow_pickle=True))

# n3 -- explore_multicompare.ipynb
POOLD      = load('multicompare_pool')
UCOS       = load('multicompare_ucos')
DIRS3      = load('multicompare_dirs')
SPACE3     = load('multicompare_space')
RF3        = load('multicompare_rf')
ATTR3      = load('multicompare_attr')
DAS15_3    = load('multicompare_das15')
LATE       = load('multicompare_late')
LATEHEAD20 = load('multicompare_latehead20')
H27CLOUD   = load('multicompare_h27l20cloud')
PCA20      = load('multicompare_pca20causal')
TWO        = load('comparator_data')          # overlap_k20: the two-number comparator neurons

# n2 -- explore_multicompare2.ipynb
DIRS2  = load('multicompare2_dirs')
SPACE2 = load('multicompare2_space')
RF2    = load('multicompare2_rf')
DAS15_2 = load('multicompare2_das15')

T, POS = int(POOLD['T']), POOLD['pos'].tolist()
LAST_POS = T - 1
print(f'T={T}  POS={POS}  LAST_POS={LAST_POS}')

## Style

In [ ]:
LABEL_FS, TICK_FS, LEGEND_FS, TITLE_FS = 20, 14, 14, 15
FS_PANEL = 20
DPI, SAVE = 300, True

COL = {'full': '#2166AC', 'das': '#C97B00', 'pc1': '#C06A9B', 'h14': '#117777', 'ceil': '#6E6E6E'}
NCOL = {1: '#C97B00', 2: '#2166AC', 3: '#2E8B57'}
Y = {1: r'$y_1$', 2: r'$y_2$', 3: r'$y_3$'}
CMAP = 'viridis'
ck = lambda s: str(s).replace(' ', '_').replace('>', 'gt')


def style_axes(ax, xlabel=None, ylabel=None, title=None, grid=True):
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_FS)
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_FS)
    if title:  ax.set_title(title, fontsize=TITLE_FS)
    ax.tick_params(labelsize=TICK_FS)
    if grid: ax.grid(alpha=0.3, axis='y')
    for side in ('top', 'right'):
        if side in ax.spines: ax.spines[side].set_visible(False)


def panel_letter(ax, s):
    ax.text(-0.16, 1.05, s, transform=ax.transAxes, fontsize=FS_PANEL,
            fontweight='bold', ha='left', va='bottom')


def savefig(fig, name):
    if SAVE:
        fig.savefig(f'{FIGS_DIR}/multicompare_{name}.png', dpi=DPI, bbox_inches='tight')


# neuron-pool bookkeeping, matching data_multicompare's NAME/POOL_LAYER/POOL_WITHIN exactly
D_FF, NEURON_LAYERS = 18944, [14, 15]
POOL_LAYER  = np.concatenate([np.full(D_FF, l) for l in NEURON_LAYERS])
POOL_WITHIN = np.concatenate([np.arange(D_FF) for _ in NEURON_LAYERS])
NAME = lambda p: f'L{POOL_LAYER[p]}#{POOL_WITHIN[p]}'


def rf_cube3(pooled_id):
    """(G, G, G) activation cube over (y1, y2, y3) for one pooled neuron, from RF3."""
    l, w = int(POOL_LAYER[pooled_id]), int(POOL_WITHIN[pooled_id])
    c = int(np.where(RF3[f'rf_idx_L{l}'] == w)[0][0])
    g = RF3['rf_grid']
    return RF3[f'rf_acts_L{l}'][:, c].reshape(len(g), len(g), len(g)), g


def rf_field2(pooled_id):
    """(G, G) activation field over (y1, y2) for one pooled neuron, from RF2."""
    l, w = int(POOL_LAYER[pooled_id]), int(POOL_WITHIN[pooled_id])
    c = int(np.where(RF2[f'rf_idx_L{l}'] == w)[0][0])
    g = RF2['rf_grid']
    return RF2[f'rf_acts_L{l}'][:, c].reshape(len(g), len(g)), g


# same diverging map as makefig_comparator's receptive fields: colourblind-safe, and not
# red/blue (already spoken for by the y1/y2/y3 palette elsewhere in this figure)
from matplotlib.colors import LinearSegmentedColormap


def trim_cmap(name, t, n=256):
    return LinearSegmentedColormap.from_list(
        f'{name}_t{t}', matplotlib.colormaps[name](np.linspace(t, 1 - t, n)))


RF_CM = trim_cmap('PRGn', 0.16)

## The figure

In [ ]:
import matplotlib.patches as mpatches

fig = plt.figure(figsize=(26, 17))
gs = fig.add_gridspec(3, 4, wspace=0.42, hspace=0.38)
GRID_BOTTOM, GRID_TOP, GRID_LEFT, GRID_RIGHT = gs.get_grid_positions(fig)

CASE_LIST3 = ['y1', 'y2', 'y3']
CASE_LIST2 = ['y1', 'y2']
LATE_GROUPS = ['s1>s2', 's1>s3', 's2>s3']
GROUP_LAB = {'s1>s2': '$y_1{>}y_2$', 's1>s3': '$y_1{>}y_3$', 's2>s3': '$y_2{>}y_3$'}
ARGMAX_SLOT3 = DIRS3['cloud_y'].argmax(1)
BIGGER2 = (DIRS2['cloud_y'][:, 1] > DIRS2['cloud_y'][:, 0]).astype(int)   # 0: y1>y2, 1: y2>y1
LETTERS = iter('abcdefghijkl')
DAS_COL = COL['pc1']    # the DAS-k1 bar in (d)/(h) -- distinct from NCOL[1] amber ("y1 pert.")
H27_COL = COL['h14']    # H27's bar in (k) -- distinct from NCOL[1] amber too


def panel_letter(row, col, s):
    # figure-coordinate placement (not axes-relative): a nested subgridspec's top-left cell
    # (panels c, g) does not sit at the same *rendered* y as a plain full-cell axis once
    # aspect='equal' images adjust their own box, so anchoring to the gridspec geometry directly
    # is what keeps every row's letters level.
    fig.text(GRID_LEFT[col] - 0.012, GRID_TOP[row] + 0.006, s, fontsize=FS_PANEL,
              fontweight='bold', ha='left', va='bottom')


def no_tr(ax):
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)


def highlight_cell(ax, r, c):
    ax.add_patch(mpatches.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False,
                                    edgecolor='black', lw=2.2))


def rf_imshow(ax, field, extent):
    fn = field / (np.abs(field).max() + 1e-9)
    ax.imshow(fn.T, origin='lower', extent=extent, cmap=RF_CM, vmin=-1, vmax=1, aspect='equal')


# ================================================================================ Row 1 (y3) ===
# --- a: orthogonality of u1, u2, v1, v2, v3 ---
ax = fig.add_subplot(gs[0, 0])
DSET1 = [(r'$\mathbf{u}_1$', UCOS['u1']), (r'$\mathbf{u}_2$', UCOS['u2']),
         (r'$\mathbf{v}_1$', DIRS3['v1']), (r'$\mathbf{v}_2$', DIRS3['v2']),
         (r'$\mathbf{v}_3$', DIRS3['v3'])]
V1 = [np.asarray(v, dtype=np.float64).reshape(-1) for _, v in DSET1]
C1 = np.array([[float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))) for b in V1] for a in V1])
im = ax.imshow(np.where(np.tril(np.ones_like(C1)) > 0, C1, np.nan),
               cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'), vmin=-1, vmax=1)
ax.set_xticks(range(len(DSET1))); ax.set_xticklabels([d[0] for d in DSET1], fontsize=TICK_FS + 2)
ax.set_yticks(range(len(DSET1))); ax.set_yticklabels([d[0] for d in DSET1], fontsize=TICK_FS + 2)
for i in range(len(DSET1)):
    for j in range(i + 1):
        ax.text(j, i, f'{C1[i, j]:+.2f}', ha='center', va='center', fontsize=9,
                color='white' if abs(C1[i, j]) > 0.6 else 'black')
for (r, c) in [(3, 2), (4, 2), (4, 3)]:      # v1/v2/v3 pairwise cross-interactions only
    highlight_cell(ax, r, c)
panel_letter(0, 0, f'({next(LETTERS)})')

# --- b: linearity -- separate patches vs the (pre-MLP) span ---
ax = fig.add_subplot(gs[0, 1])
ARMS1B = [('v1 only', r'$\mathbf{v}_1$'), ('v2 only', r'$\mathbf{v}_2$'),
          ('v3 only', r'$\mathbf{v}_3$'),
          ('v1+v2+v3 sep', r'$\mathbf{v}_1$&$\mathbf{v}_2$&$\mathbf{v}_3$'),
          ('span preMLP', r'$\mathbf{v}_1,\mathbf{v}_2,\mathbf{v}_3$ space'),
          ('L14 full', 'full')]
x, w = np.arange(len(ARMS1B)), 0.8 / len(CASE_LIST3)
for ci, case in enumerate(CASE_LIST3):
    j = int(case[1])
    vals = [SPACE3[f'sp_posrec_{case}_{ck(nm)}'] for nm, _ in ARMS1B]
    xp = x + (ci - (len(CASE_LIST3) - 1) / 2) * w
    ax.bar(xp, [v.mean() for v in vals], w, yerr=[v.std() / np.sqrt(len(v)) for v in vals],
           capsize=2.5, color=NCOL[j], alpha=0.9, label=f'{Y[j]} pert.')
ax.axvline(2.5, color='0.75', lw=1, ls='-')      # individual directions | combined patches
ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in ARMS1B], fontsize=TICK_FS - 2,
                  rotation=45, ha='right')
ax.axhline(0, color='0.6', lw=0.8)
style_axes(ax, ylabel='PR', grid=False)
ax.legend(fontsize=LEGEND_FS, frameon=False)
panel_letter(0, 1, f'({next(LETTERS)})')

# --- c: local (L14#5076) / global (L15#12784) comparator neurons, all three marginals each ---
D_FF = 18944
LOCAL_ID, GLOBAL_ID = 5076, D_FF + 12784   # L14#5076, L15#12784
NEURONS_C1 = [(LOCAL_ID, 'local\nL14#5076'), (GLOBAL_ID, 'global\nL15#12784')]
MARGIN_TRIOS = [(2, '$y_1$', '$y_2$'), (1, '$y_1$', '$y_3$'), (0, '$y_2$', '$y_3$')]
inner = gs[0, 2].subgridspec(2, 3, wspace=0.40, hspace=0.25)
for ri, (pooled, label) in enumerate(NEURONS_C1):
    cube, g3 = rf_cube3(pooled)
    ext3 = [g3[0], g3[-1], g3[0], g3[-1]]
    for ci, (avg_axis, xl, yl) in enumerate(MARGIN_TRIOS):
        ax = fig.add_subplot(inner[ri, ci])
        rf_imshow(ax, cube.mean(axis=avg_axis), ext3)
        ax.set_xlabel(xl, fontsize=LABEL_FS - 3, labelpad=1)
        ax.set_ylabel(yl, fontsize=LABEL_FS - 3, labelpad=1)
        ax.tick_params(labelsize=6)
        if ci == 0:
            ax.text(-1.05, 0.5, label, transform=ax.transAxes, ha='center', va='center',
                    fontsize=TICK_FS - 2)
panel_letter(0, 2, f'({next(LETTERS)})')

# --- d: comparator direction (y3 max vs not), PR as inset ---
ax = fig.add_subplot(gs[0, 3])
p1 = DAS15_3['proj_y3_k1'][:, 0]
Y3_MAX = (ARGMAX_SLOT3 == 2)
bins = np.linspace(p1.min(), p1.max(), 40)
ax.hist(p1[~Y3_MAX], bins=bins, color=COL['ceil'], alpha=0.6, label='$y_3$ not max')
ax.hist(p1[Y3_MAX], bins=bins, color=NCOL[3], alpha=0.6, label='$y_3$ max')
style_axes(ax, xlabel='L15 DAS $k$=1 (fit $y_3$)', ylabel='prompts', grid=False)
ax.legend(fontsize=LEGEND_FS - 1, loc='upper right', frameon=False)
panel_letter(0, 3, f'({next(LETTERS)})')

axi = ax.inset_axes([0.34, 0.60, 0.28, 0.32])
bars = [('DAS', DAS15_3['posrec_y3_from_y3_k1'], DAS_COL),
        ('full', DAS15_3['posrec_y3_full'], COL['ceil'])]
axi.bar(range(len(bars)), [v.mean() for _, v, _ in bars],
        yerr=[v.std() / np.sqrt(len(v)) for _, v, _ in bars],
        color=[c for _, _, c in bars], capsize=2, alpha=0.9)
axi.set_xticks(range(len(bars))); axi.set_xticklabels([nm for nm, _, _ in bars], fontsize=7)
axi.tick_params(labelsize=7); axi.set_ylabel('PR', fontsize=7); no_tr(axi)

# ================================================================================ Row 2 (y2) ===
# --- e: orthogonality of u1, u2, w1, w2 ---
ax_row1 = ax = fig.add_subplot(gs[1, 0])
DSET2 = [(r'$\mathbf{u}_1$', UCOS['u1']), (r'$\mathbf{u}_2$', UCOS['u2']),
         (r'$\mathbf{w}_1$', DIRS2['w1']), (r'$\mathbf{w}_2$', DIRS2['w2'])]
V2 = [np.asarray(v, dtype=np.float64).reshape(-1) for _, v in DSET2]
C2 = np.array([[float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))) for b in V2] for a in V2])
im = ax.imshow(np.where(np.tril(np.ones_like(C2)) > 0, C2, np.nan),
               cmap=matplotlib.colormaps['RdBu_r'].with_extremes(bad='white'), vmin=-1, vmax=1)
ax.set_xticks(range(len(DSET2))); ax.set_xticklabels([d[0] for d in DSET2], fontsize=TICK_FS + 2)
ax.set_yticks(range(len(DSET2))); ax.set_yticklabels([d[0] for d in DSET2], fontsize=TICK_FS + 2)
for i in range(len(DSET2)):
    for j in range(i + 1):
        ax.text(j, i, f'{C2[i, j]:+.2f}', ha='center', va='center', fontsize=9,
                color='white' if abs(C2[i, j]) > 0.6 else 'black')
highlight_cell(ax, 3, 2)       # w1, w2 cross-interaction only
panel_letter(1, 0, f'({next(LETTERS)})')

# --- f: linearity at y2 ---
ax = fig.add_subplot(gs[1, 1])
ARMS2B = [('w1 only', r'$\mathbf{w}_1$'), ('w2 only', r'$\mathbf{w}_2$'),
          ('w1+w2 sep', r'$\mathbf{w}_1$&$\mathbf{w}_2$'),
          ('span preMLP', r'$\mathbf{w}_1,\mathbf{w}_2$ plane'), ('L14 full', 'full')]
x, w = np.arange(len(ARMS2B)), 0.8 / len(CASE_LIST2)
for ci, case in enumerate(CASE_LIST2):
    j = int(case[1])
    vals = [SPACE2[f'sp_posrec_{case}_{ck(nm)}'] for nm, _ in ARMS2B]
    xp = x + (ci - (len(CASE_LIST2) - 1) / 2) * w
    ax.bar(xp, [v.mean() for v in vals], w, yerr=[v.std() / np.sqrt(len(v)) for v in vals],
           capsize=2.5, color=NCOL[j], alpha=0.9, label=f'{Y[j]} pert.')
ax.axvline(1.5, color='0.75', lw=1, ls='-')      # individual directions | combined patches
ax.set_xticks(x); ax.set_xticklabels([lab for _, lab in ARMS2B], fontsize=TICK_FS - 2,
                  rotation=45, ha='right')
ax.axhline(0, color='0.6', lw=0.8)
style_axes(ax, ylabel='PR', grid=False)
ax.legend(fontsize=LEGEND_FS, frameon=False)
panel_letter(1, 1, f'({next(LETTERS)})')

# --- g: receptive fields at y2 -- same local/global neurons as panel c ---
inner = gs[1, 2].subgridspec(1, 2, wspace=0.32)
for ci, (pooled, label) in enumerate(NEURONS_C1):
    ax = fig.add_subplot(inner[0, ci])
    field2, g2 = rf_field2(pooled)
    rf_imshow(ax, field2, [g2[0], g2[-1], g2[0], g2[-1]])
    ax.set_xlabel('$y_1$', fontsize=LABEL_FS - 1); ax.set_ylabel('$y_2$', fontsize=LABEL_FS - 1)
    ax.tick_params(labelsize=8)
    ax.text(0.5, 1.08, label.replace(chr(10), ' '), transform=ax.transAxes, ha='center',
            va='bottom', fontsize=TICK_FS - 1)
panel_letter(1, 2, f'({next(LETTERS)})')

# --- h: comparator direction at y2 (max vs not), PR as inset ---
ax = fig.add_subplot(gs[1, 3])
p1_2 = DAS15_2['proj_y2_k1'][:, 0]
Y2_MAX = (BIGGER2 == 1)
bins = np.linspace(p1_2.min(), p1_2.max(), 40)
ax.hist(p1_2[~Y2_MAX], bins=bins, color=COL['ceil'], alpha=0.6, label='$y_2$ not max')
ax.hist(p1_2[Y2_MAX], bins=bins, color=NCOL[2], alpha=0.6, label='$y_2$ max')
style_axes(ax, xlabel='L15 DAS $k$=1 (fit $y_2$)', ylabel='prompts', grid=False)
ax.legend(fontsize=LEGEND_FS - 1, loc='upper right', frameon=False)
panel_letter(1, 3, f'({next(LETTERS)})')

axi = ax.inset_axes([0.34, 0.60, 0.28, 0.32])
bars2 = [('DAS', DAS15_2['posrec_y2_from_y2_k1'], DAS_COL),
         ('full', DAS15_2['posrec_y2_full'], COL['ceil'])]
axi.bar(range(len(bars2)), [v.mean() for _, v, _ in bars2],
        yerr=[v.std() / np.sqrt(len(v)) for _, v, _ in bars2],
        color=[c for _, _, c in bars2], capsize=2, alpha=0.9)
axi.set_xticks(range(len(bars2))); axi.set_xticklabels([nm for nm, _, _ in bars2], fontsize=7)
axi.tick_params(labelsize=7); axi.set_ylabel('PR', fontsize=7); no_tr(axi)

# ======================================================================= Row 3 (last token) ===
# --- i: the answer in L20's output -- 2D PCA coloured by max position ---
ax_row2 = ax = fig.add_subplot(gs[2, 0])
MAXSLOT_LATE = LATE['late_y'].argmax(1) + 1
pcL20 = LATE['pca_coords_L20']
colors_late = [NCOL[s] for s in MAXSLOT_LATE]
ax.scatter(pcL20[:, 0], pcL20[:, 1], color=colors_late, s=6, alpha=0.65, edgecolors='none')
style_axes(ax, xlabel='PC1', ylabel='PC2', grid=False)
for slot in (1, 2, 3):
    ax.scatter([], [], color=NCOL[slot], label=f'max $y_{slot}$')
ax.legend(fontsize=LEGEND_FS - 1, frameon=False, loc='lower right')
panel_letter(2, 0, f'({next(LETTERS)})')

# --- j: H27 attention (question only, example trimmed) ---
ax = fig.add_subplot(gs[2, 1])
Q_START = 30    # first token of "The maximum of ..." -- the one-shot example ends just before
MAXSLOT27 = H27CLOUD['y'].argmax(1) + 1
xs = np.arange(Q_START, T)
for slot in (1, 2, 3):
    a = H27CLOUD['attn'][MAXSLOT27 == slot][:, Q_START:]
    ax.plot(xs, a.mean(0), color=NCOL[slot], lw=1.8, label=f'max $y_{slot}$')
for p in POS:
    ax.axvline(p, color='0.6', ls=':', lw=1)
ax.axvline(LAST_POS, color='0.3', ls='--', lw=1.1)
ax.set_xlim(Q_START - 0.5, LAST_POS + 0.5)
ax.set_xticks(POS + [LAST_POS])
ax.set_xticklabels(['$y_1$', '$y_2$', '$y_3$', 'is'], fontsize=TICK_FS - 1)
style_axes(ax, xlabel='token position', ylabel='H27 attention', grid=False)
ax.legend(fontsize=LEGEND_FS, frameon=False, loc='upper left')
panel_letter(2, 1, f'({next(LETTERS)})')

# --- k: position recovery of every L20 head (own output interchanged), H27 highlighted ---
ax = fig.add_subplot(gs[2, 2])
pr = LATEHEAD20[f'posrec_{ck("s1>s2")}']
order_ix = np.argsort(-pr.mean(1))
h27_pos = int(np.where(order_ix == 27)[0][0])
colors_h = [H27_COL if h == 27 else COL['ceil'] for h in order_ix]
ax.bar(np.arange(len(order_ix)), pr.mean(1)[order_ix], color=colors_h, alpha=0.9)
ax.set_xticks([h27_pos]); ax.set_xticklabels(['H27']); ax.axhline(0, color='0.6', lw=0.6)
style_axes(ax, xlabel='attn. heads (L20)', ylabel='PR', grid=False)
panel_letter(2, 2, f'({next(LETTERS)})')

# --- l: is the 2D PCA subspace causal? full-rank vs. 2D-PCA-subspace interchange ---
ax = fig.add_subplot(gs[2, 3])
x3b = np.arange(len(LATE_GROUPS))
full = [LATE[f'resid_posrec_{ck(g)}_L20'] for g in LATE_GROUPS]
pca_v = [PCA20[f'posrec_{ck(g)}'] for g in LATE_GROUPS]
w3b = 0.35
ax.bar(x3b - w3b / 2, [v.mean() for v in full], w3b, color=COL['full'], alpha=0.9, label='full-rank')
ax.bar(x3b + w3b / 2, [v.mean() for v in pca_v], w3b, color=COL['pc1'], alpha=0.9, label='2D PCA')
ax.set_xticks(x3b); ax.set_xticklabels([GROUP_LAB[g] for g in LATE_GROUPS], fontsize=TICK_FS)
style_axes(ax, ylabel='PR', grid=False)
ax.legend(fontsize=LEGEND_FS, frameon=False)
panel_letter(2, 3, f'({next(LETTERS)})')

# ---------------------------------------------------------------------- big row labels -----
ROW_LABELS = ['$y_3$ position', '$y_2$ position', 'last token']
for row, label in enumerate(ROW_LABELS):
    y_center = 0.5 * (GRID_BOTTOM[row] + GRID_TOP[row])
    fig.text(GRID_LEFT[0] - 0.045, y_center, label, rotation=90, ha='center', va='center',
              fontsize=FS_PANEL, fontweight='bold')

savefig(fig, 'appendix_summary')
# Frozen for the appendix (Figure app-k3-summary); the main-paper version lives in makefig_multicompare.ipynb.
plt.savefig('./figs_appendix/app_k3_summary.pdf', bbox_inches='tight')
plt.show()